# The Deutsch and Deutsch-Jozsa Algorithms with Qiskit

**Qiskit Fall Fest workshop notebook.** Written for Qiskit 2.x, runs in Google Colab, no account needed.

This is the first *algorithm* notebook of the week. It follows *Quantum Computing Essentials with Qiskit* and assumes you know gates (Essentials, Chapter 3), measurement and the Sampler (Chapter 4), and **interference** (Chapter 7). The half adder in Essentials Chapter 10 ended with a promise: to get a real advantage, use interference so that a *global* property of a function survives the measurement. This notebook keeps that promise.

---

## How to read this notebook

Same rhythm as before. Each chapter follows three steps:

1. We **look** at something in code (a truth table, a circuit, a set of Bloch spheres, a histogram).
2. We **explain** the mathematics behind what we just saw.
3. We **stop and ask you to try** something yourself.

The same conventions are used throughout:

> **Predict.** Before running a cell, write down what you expect to see. Being wrong is the useful part.

> **Physics note.** A short historical or physical aside that connects the code to the real laboratory.

> **Check.** A cell that tests your own answer and tells you whether it is right.

Exercises have hints and full solutions hidden inside collapsible blocks. Open them only after a real attempt.

## The story in one paragraph

We are handed a function as a black box and asked a yes/no question about it: does it give the same answer for every input, or the same answer for exactly half of them? Classically we must open the box many times. Deutsch (1985) showed that for a one-bit function a quantum computer needs to open it **once** instead of twice. Deutsch and Jozsa (1992) extended this to $n$-bit functions, where the classical cost grows exponentially and the quantum cost stays at **one**. We build Deutsch's algorithm first, understand exactly why it works, and then add Jozsa's generalisation on top. We finish with a close cousin, the Bernstein-Vazirani algorithm.

## Contents

| Chapter | Topic |
|---|---|
| 0 | Setting up |
| 1 | The problem: questions about a black box |
| 2 | Oracles: functions as reversible circuits |
| 3 | Phase kickback: from bit oracle to phase oracle |
| 4 | Deutsch's algorithm |
| 5 | Going to $n$ bits: the promise problem |
| 6 | The mathematics of Deutsch-Jozsa |
| 7 | Building $n$-bit oracles |
| 8 | Running Deutsch-Jozsa, and what happens when the promise breaks |
| 9 | Noise and real devices |
| 10 | A worked problem: Bernstein-Vazirani |
| | Where we go next, and further reading |

## Chapter 0. Setting up

The same three packages as in the Essentials notebook: `qiskit`, `qiskit-aer` and `qiskit-ibm-runtime`. Run the next cell once.

(Working locally instead of Colab? With `uv` you can skip the install cell and start Jupyter with everything in one line: `uv run --with "qiskit[visualization]" --with qiskit-aer --with qiskit-ibm-runtime --with pylatexenc --with ipywidgets --with jupyterlab jupyter lab`.)

In [ ]:
%pip install -q "qiskit[visualization]" qiskit-aer qiskit-ibm-runtime pylatexenc ipywidgets

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt

import qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.circuit.library import ZGate
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit.primitives import StatevectorSampler
from ipywidgets import interact, IntSlider, Dropdown

print("Qiskit version:", qiskit.__version__)

sampler = StatevectorSampler()


# The two helpers from the Essentials notebook.

def fig_to_img(fig):
    # Render a matplotlib Figure to an image array so we can place
    # several Bloch-sphere figures side by side.
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    buf.seek(0)
    return plt.imread(buf)


def check(label, test):
    # Run a test (a function returning True/False) and report the result.
    # If your variable is still None or the code raises, we say "Not yet".
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("Correct:  " if ok else "Not yet:  ") + label)

## Chapter 1. The problem: questions about a black box

Someone gives us a function $f:\{0,1\}\to\{0,1\}$, one bit in, one bit out. We cannot look inside. We can only **query** it: hand it an input and read the output. Each query costs us something (think of an expensive experiment, or a slow subroutine), so we want as few as possible.

There are exactly four such functions.

| name | $f(0)$ | $f(1)$ | type |
|---|---|---|---|
| `const_0` | 0 | 0 | constant |
| `const_1` | 1 | 1 | constant |
| `identity` | 0 | 1 | balanced |
| `negation` | 1 | 0 | balanced |

A function is **constant** if it gives the same output on every input, and **balanced** if it outputs 0 on exactly half of the inputs and 1 on the other half.

**Deutsch's problem.** Decide whether $f$ is constant or balanced.

Classically, one query is never enough. If you learn $f(0) = 0$, the function could be `const_0` or `identity`, one of each type. You must also query $f(1)$. **Two queries, always.**

Notice the question is about a *global* property: is $f(0) = f(1)$? Equivalently, what is $f(0)\oplus f(1)$? We do not need to know $f(0)$ or $f(1)$ individually, only whether they agree. That is the opening a quantum computer will use.

> **Physics note.** David Deutsch posed this problem in his 1985 paper *Quantum theory, the Church-Turing principle and the universal quantum computer*, the paper that defined the quantum Turing machine. His original algorithm only succeeded half the time, and it was not until 1998 that Cleve, Ekert, Macchiavello and Mosca gave the deterministic version we build here. A tiny problem, chosen deliberately: the point was to show that a quantum computer can do *something* with fewer steps than any classical one.

Here is the classical approach in code, written as a "query counter" so the cost is visible.

In [ ]:
functions_1bit = {
    "const_0":  lambda x: 0,
    "const_1":  lambda x: 1,
    "identity": lambda x: x,
    "negation": lambda x: 1 - x,
}

def classical_deutsch(f):
    queries = 0
    a = f(0); queries += 1
    b = f(1); queries += 1
    return ("constant" if a == b else "balanced"), queries

for name, f in functions_1bit.items():
    verdict, q = classical_deutsch(f)
    print(f"{name:9s} -> {verdict:9s} using {q} queries")

### Exercise 1: classifying functions

**Goal.** Below are four functions of **two** bits, given as truth tables in the order $f(00), f(01), f(10), f(11)$. Put the names of the **balanced** ones into the list `ex1_balanced`.

| name | truth table |
|---|---|
| `xor` | 0, 1, 1, 0 |
| `and_` | 0, 0, 0, 1 |
| `first_bit` | 0, 0, 1, 1 |
| `always_one` | 1, 1, 1, 1 |

<details>
<summary>Hint</summary>

Balanced means exactly two 0s and two 1s. One of the four is neither constant nor balanced. Keep it in mind for Chapter 8.
</details>

In [ ]:
ex1_balanced = None       # e.g. ["xor", ...]

In [ ]:
check("the balanced functions", lambda: sorted(ex1_balanced) == ["first_bit", "xor"])

<details>
<summary>Solution</summary>

```python
ex1_balanced = ["xor", "first_bit"]
```

`always_one` is constant. `and_` has three 0s and one 1, so it is neither. The Deutsch-Jozsa algorithm is only *promised* to work on functions that are constant or balanced, and we will see in Chapter 8 what it does with `and_`.
</details>

Further reading: the first lesson of [Fundamentals of quantum algorithms](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms) sets up the query model of computation in exactly this way.

## Chapter 2. Oracles: functions as reversible circuits

To query $f$ with a quantum computer, $f$ has to be a quantum gate, which means a unitary, which means **reversible**. But `const_0` is not reversible: from the output 0 you cannot tell whether the input was 0 or 1.

The fix is the same one we used for the half adder in Essentials Chapter 10. Keep the input, and XOR the answer into a fresh qubit:

$$U_f\,|x\rangle|y\rangle = |x\rangle\,|y\oplus f(x)\rangle .$$

This is called the **bit oracle** (or XOR oracle) for $f$. Applying it twice gives back $|x\rangle|y\rangle$, because $y\oplus f(x)\oplus f(x) = y$. So $U_f$ is its own inverse, and in particular reversible, for *any* $f$.

**Qubit convention for this whole notebook.** The input $x$ lives on qubits $0,\dots,n-1$ and the output qubit $y$ is the last one, qubit $n$. For Deutsch's problem, $n = 1$: qubit 0 is $x$, qubit 1 is $y$.

Here are the four one-bit oracles.

| function | circuit | why |
|---|---|---|
| `const_0` | nothing | $y \oplus 0 = y$ |
| `const_1` | X on $y$ | $y \oplus 1 = \bar y$ |
| `identity` | CNOT from $x$ to $y$ | $y \oplus x$ |
| `negation` | CNOT, then X on $y$ | $y \oplus x \oplus 1$ |

In [ ]:
def deutsch_oracle(name):
    qc = QuantumCircuit(2, name=f"U_f ({name})")      # qubit 0 = x, qubit 1 = y
    if name == "const_1":
        qc.x(1)
    elif name == "identity":
        qc.cx(0, 1)
    elif name == "negation":
        qc.cx(0, 1)
        qc.x(1)
    return qc

fig, axes = plt.subplots(1, 4, figsize=(14, 2.2))
for ax, name in zip(axes, functions_1bit):
    deutsch_oracle(name).draw("mpl", ax=ax)
    ax.set_title(name)
plt.show()

A circuit is only as good as its tests. The helper below reads the truth table back out of *any* oracle by preparing $|x\rangle|0\rangle$ for every input and looking at the output qubit. We will reuse it for every oracle in the notebook.

In [ ]:
def truth_table(oracle, n):
    # Return [f(0), f(1), ..., f(2^n - 1)] for an oracle on n input qubits + 1 output qubit.
    table = []
    for x in range(2**n):
        qc = QuantumCircuit(n + 1)
        for i in range(n):
            if (x >> i) & 1:
                qc.x(i)                               # prepare |x> (little-endian)
        qc.compose(oracle, inplace=True)
        outcome = int(np.argmax(Statevector(qc).probabilities()))
        table.append((outcome >> n) & 1)              # the output qubit is the top bit
    return table

for name in functions_1bit:
    print(f"{name:9s}  f(0), f(1) = {truth_table(deutsch_oracle(name), 1)}")

Each oracle is a permutation of the four basis states, so its matrix contains only 0s and 1s. For example, the `identity` oracle is the CNOT matrix from Essentials Chapter 6.

In [ ]:
for name in functions_1bit:
    print(f"--- {name} ---")
    print(Operator(deutsch_oracle(name)).data.real.astype(int))

> **Predict.** These matrices are in Qiskit's basis order $|y\,x\rangle = |00\rangle, |01\rangle, |10\rangle, |11\rangle$. Which two basis states does the `identity` oracle swap? Check against the printout.

### Exercise 2: an oracle for AND

**Goal.** Build the oracle `ex2_oracle` for the two-bit function $f(x_1x_0) = x_0 \wedge x_1$ (the `and_` function from Exercise 1). It acts on three qubits: inputs on qubits 0 and 1, output on qubit 2.

<details>
<summary>Hint</summary>

You need to flip the output only when *both* inputs are 1. Essentials Chapter 10 introduced a gate that does exactly that.
</details>

In [ ]:
ex2_oracle = None

In [ ]:
check("ex2_oracle computes AND", lambda: truth_table(ex2_oracle, 2) == [0, 0, 0, 1])

<details>
<summary>Solution</summary>

```python
ex2_oracle = QuantumCircuit(3, name="U_and")
ex2_oracle.ccx(0, 1, 2)          # Toffoli: flip qubit 2 iff qubits 0 and 1 are both 1
```
</details>

Further reading: [Construct circuits](https://quantum.cloud.ibm.com/docs/en/guides/construct-circuits) for the gates used here.

## Chapter 3. Phase kickback: from bit oracle to phase oracle

The bit oracle writes $f(x)$ into the output qubit as a *bit*. Measuring that bit after one query tells us one value $f(x)$, which is no better than classical. The trick is to prepare the output qubit not in $|0\rangle$ but in

$$|-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt2}.$$

Watch what happens. If $f(x) = 0$, the oracle does nothing. If $f(x) = 1$, it applies X to the output qubit, and $X|-\rangle = \frac{|1\rangle - |0\rangle}{\sqrt2} = -|-\rangle$. Either way,

$$U_f\,|x\rangle|-\rangle = (-1)^{f(x)}\,|x\rangle|-\rangle .$$

The output qubit is unchanged. The function value has turned into a **sign** in front of the input state. This is **phase kickback**: $|-\rangle$ is an eigenstate of X with eigenvalue $-1$, and that eigenvalue is "kicked back" onto the input.

With the output qubit fixed in $|-\rangle$ we can forget about it and describe the oracle purely by its action on the input,

$$O_f\,|x\rangle = (-1)^{f(x)}\,|x\rangle ,$$

which is called the **phase oracle**. It is a diagonal matrix with $\pm1$ on the diagonal.

On its own, a sign in front of a basis state is a global phase and is invisible. But when the input is in a **superposition**, the signs become relative phases between branches, and interference (Essentials Chapter 7) can turn relative phases into measurement outcomes.

> **Physics note.** The input qubit is called the "control" of the CNOT, and we tend to imagine that controls are never changed. Phase kickback shows that this picture is basis-dependent. In the X basis, a CNOT's roles are reversed: the "target" controls the "control". Essentials Chapter 3 showed $HZH = X$; the same identity, applied to two qubits, gives $(H\otimes H)\,\text{CNOT}\,(H\otimes H) = $ a CNOT pointing the other way.

Let us see the kickback on the Bloch sphere. Input qubit 0 starts in $|+\rangle$, output qubit 1 in $|-\rangle$, and the oracle is `identity` (a CNOT).

In [ ]:
def show_before_after(prep, oracle, title=""):
    before = Statevector(prep)
    after = Statevector(prep.compose(oracle))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, (t, sv) in zip(axes, [("BEFORE the oracle", before), ("AFTER the oracle", after)]):
        f = plot_bloch_multivector(sv, title=t)
        ax.imshow(fig_to_img(f)); ax.axis("off"); plt.close(f)
    if title:
        fig.suptitle(title, y=0.9)
    plt.show()

prep = QuantumCircuit(2)
prep.h(0)                 # input  |+>
prep.x(1); prep.h(1)      # output |->
show_before_after(prep, deutsch_oracle("identity"), "identity oracle, output qubit in |->")

The **input** qubit (left) has swung from $+x$ to $-x$: the state $\frac{|0\rangle+|1\rangle}{\sqrt2}$ became $\frac{|0\rangle-|1\rangle}{\sqrt2}$ because only the $|1\rangle$ branch picked up the sign. The output qubit (right) has not moved.

> **Predict.** Repeat with the `const_1` oracle. Both branches pick up $-1$. What happens to the input qubit now?

In [ ]:
show_before_after(prep, deutsch_oracle("const_1"), "const_1 oracle, output qubit in |->")

Nothing visible. Both branches got the same sign, which is a global phase. **A constant function produces no relative phase; a balanced one does.** That one sentence is Deutsch's algorithm. The rest is reading the relative phase out.

Here are the phase oracles for all four functions, written as the signs $\big((-1)^{f(0)}, (-1)^{f(1)}\big)$ they put on the input.

In [ ]:
for name, f in functions_1bit.items():
    signs = [(-1)**f(x) for x in (0, 1)]
    kind = "constant" if signs[0] == signs[1] else "balanced"
    print(f"{name:9s}  signs on |0>, |1> = {signs}   -> relative sign {signs[0]*signs[1]:+d}  ({kind})")

### Exercise 3: a phase oracle is a familiar gate

**Goal.** With the output qubit in $|-\rangle$, the `negation` oracle acts on the input as $O_f|x\rangle = (-1)^{1-x}|x\rangle$. That is a $2\times2$ diagonal matrix. Build a one-qubit circuit `ex3_circuit` containing a **single standard gate** that equals it, up to a global phase.

<details>
<summary>Hint</summary>

Write out the diagonal: $(-1)^{1-0} = -1$ and $(-1)^{1-1} = +1$. Which gate from Essentials Chapter 3 has the diagonal $(1, -1)$? How does it differ from $(-1, 1)$?
</details>

In [ ]:
ex3_circuit = None

In [ ]:
check("the negation phase oracle equals your gate up to global phase",
      lambda: Operator(ex3_circuit).equiv(Operator(np.diag([-1, 1]))) and ex3_circuit.size() == 1)

<details>
<summary>Solution</summary>

```python
ex3_circuit = QuantumCircuit(1)
ex3_circuit.z(0)
```

The phase oracle is $\mathrm{diag}(-1, 1) = -Z$. The overall $-1$ is a global phase and changes nothing. The `identity` oracle gives $\mathrm{diag}(1,-1) = Z$ exactly. So both balanced functions act as a Z gate on the input, and both constant ones act as the identity (times $\pm1$). Deutsch's problem has become: "is the hidden gate $I$ or $Z$?", and Essentials Chapter 7 already showed how to tell those apart.
</details>

Further reading: R. Cleve, A. Ekert, C. Macchiavello and M. Mosca, [Quantum algorithms revisited](https://arxiv.org/abs/quant-ph/9708016) (1998), which introduced the phase-kickback view of these algorithms.

## Chapter 4. Deutsch's algorithm

Put the pieces together.

1. Start in $|0\rangle_x|1\rangle_y$.
2. Apply H to both qubits: the input becomes $|+\rangle$, the output becomes $|-\rangle$.
3. Query the oracle **once**.
4. Apply H to the input and measure it.

### The mathematics, step by step

We write the input qubit first in the kets below, $|x\rangle|y\rangle$, to keep the algebra readable. (Qiskit prints them the other way round.)

$$|\psi_0\rangle = |0\rangle|1\rangle$$

After the Hadamards:

$$|\psi_1\rangle = \frac{|0\rangle+|1\rangle}{\sqrt2}\otimes\frac{|0\rangle-|1\rangle}{\sqrt2} = \frac{1}{\sqrt2}\sum_{x\in\{0,1\}}|x\rangle\;\otimes|-\rangle .$$

After one query, by Chapter 3:

$$|\psi_2\rangle = \frac{1}{\sqrt2}\Big((-1)^{f(0)}|0\rangle + (-1)^{f(1)}|1\rangle\Big)\otimes|-\rangle .$$

Both values of $f$ are now in the state, each as a sign on its own branch. That is one query evaluating $f$ "on both inputs". But measuring now would give 0 or 1 at random and tell us nothing. We need interference. Apply H to the input, using $H|0\rangle = \frac{|0\rangle+|1\rangle}{\sqrt2}$ and $H|1\rangle = \frac{|0\rangle-|1\rangle}{\sqrt2}$:

$$|\psi_3\rangle = \left[\frac{(-1)^{f(0)} + (-1)^{f(1)}}{2}\,|0\rangle \;+\; \frac{(-1)^{f(0)} - (-1)^{f(1)}}{2}\,|1\rangle\right]\otimes|-\rangle .$$

Now read off the two cases.

- **Constant**, $f(0) = f(1)$: the signs agree, the $|1\rangle$ amplitude is $0$, and the $|0\rangle$ amplitude is $\pm1$. We measure **0** with certainty.
- **Balanced**, $f(0) \ne f(1)$: the signs disagree, the $|0\rangle$ amplitude cancels, and we measure **1** with certainty.

Compactly, $|\psi_3\rangle = \pm\,|f(0)\oplus f(1)\rangle\otimes|-\rangle$. **The measurement returns $f(0)\oplus f(1)$ directly, after one query.**

> **Physics note.** This is a Mach-Zehnder interferometer. The first Hadamard is a beam splitter that sends the input into two "arms", $|0\rangle$ and $|1\rangle$. The oracle puts a phase shift of $0$ or $\pi$ in each arm. The second Hadamard recombines the arms. If both arms got the same shift, the light exits through one port; if they differ by $\pi$, it exits through the other. The detector does not learn *which* shift was in each arm, only whether they were equal. Deutsch's problem asks precisely that. The first experimental demonstrations, in 1998, used nuclear spins in NMR (Jones and Mosca at Oxford, Chuang and co-workers at IBM and Stanford).

We write the circuit builder for general $n$ straight away, because Deutsch's algorithm will turn out to be its $n = 1$ case.

In [ ]:
def dj_circuit(oracle, n):
    # Deutsch-Jozsa circuit: n input qubits (0..n-1), one output qubit (n).
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)                    # output qubit to |1>
    qc.h(range(n + 1))         # inputs to |+>, output to |->
    qc.barrier()
    qc.compose(oracle, range(n + 1), inplace=True)   # ONE query
    qc.barrier()
    qc.h(range(n))             # interfere the input branches
    qc.measure(range(n), range(n))
    return qc

deutsch = dj_circuit(deutsch_oracle("identity"), 1)
deutsch.draw("mpl")

> **Predict.** For each of the four oracles, which outcome will we measure, and how often out of 1000 shots?

In [ ]:
results = {}
for name in functions_1bit:
    counts = sampler.run([dj_circuit(deutsch_oracle(name), 1)], shots=1000).result()[0].data.c.get_counts()
    results[name] = counts
    verdict = "constant" if counts.get("0", 0) == 1000 else ("balanced" if counts.get("1", 0) == 1000 else "??")
    print(f"{name:9s}  counts = {counts}   ->  {verdict}")

plot_histogram(list(results.values()), legend=list(results.keys()), title="Deutsch's algorithm, one query each")

Every run of every function is decided correctly, with a single query. Classically we always needed two.

### Following the state through the circuit

The `Statevector` lets us check the algebra at each step. Below we print the input qubit's amplitudes after step 3 (the query) and after step 4 (the final Hadamard), for each function. Compare with $|\psi_2\rangle$ and $|\psi_3\rangle$ above. (We factor out the output qubit, which is always $|-\rangle$.)

In [ ]:
minus = np.array([1, -1]) / np.sqrt(2)

def input_amplitudes(qc):
    # Amplitudes of the input qubit, given the output qubit is in |-> (qubit 1 = left factor).
    sv = Statevector(qc).data.reshape(2, 2)          # rows: output qubit, columns: input qubit
    return minus.conj() @ sv

for name in functions_1bit:
    qc = QuantumCircuit(2)
    qc.x(1); qc.h([0, 1])
    qc.compose(deutsch_oracle(name), inplace=True)
    after_query = input_amplitudes(qc)
    qc.h(0)
    after_h = input_amplitudes(qc)
    print(f"{name:9s} after query: {np.round(after_query * np.sqrt(2), 3)} / sqrt2    "
          f"after final H: {np.round(after_h, 3)}")

### Interactive: choose a function

Pick a function from the menu. You see the input qubit's Bloch sphere after the query (on the equator, at $+x$ or $-x$), and the measurement probabilities after the final Hadamard.

In [ ]:
@interact(name=Dropdown(options=list(functions_1bit), value="identity", description="f"))
def deutsch_explorer(name):
    qc = QuantumCircuit(2)
    qc.x(1); qc.h([0, 1])
    qc.compose(deutsch_oracle(name), inplace=True)
    fig = plot_bloch_multivector(Statevector(qc), title=f"after the query, f = {name}")
    display(fig); plt.close(fig)
    qc.h(0)
    p = Statevector(qc).probabilities([0])
    print(f"P(measure 0) = {p[0]:.3f}   P(measure 1) = {p[1]:.3f}   ->  "
          + ("constant" if np.isclose(p[0], 1) else "balanced"))

### Exercise 4: why the output qubit must be in $|-\rangle$

**Goal.** Build a Deutsch circuit `ex4_circuit` for the **balanced** `identity` oracle, but prepare the output qubit in $|+\rangle$ instead of $|-\rangle$ (that is, leave out the `x(1)`). Measure qubit 0 into one classical bit. The check confirms that this broken version always answers "constant", which is wrong. Then explain in a comment why.

<details>
<summary>Hint</summary>

Start from `QuantumCircuit(2, 1)`. Apply `h` to both qubits, compose the oracle, `h` on qubit 0, measure qubit 0. What is $X|+\rangle$?
</details>

In [ ]:
ex4_circuit = None

In [ ]:
check("with the output in |+>, a balanced function is reported as constant",
      lambda: sampler.run([ex4_circuit], shots=500).result()[0].data.c.get_counts() == {"0": 500})

<details>
<summary>Solution</summary>

```python
ex4_circuit = QuantumCircuit(2, 1)
ex4_circuit.h([0, 1])                                  # output qubit in |+>, not |->
ex4_circuit.compose(deutsch_oracle("identity"), inplace=True)
ex4_circuit.h(0)
ex4_circuit.measure(0, 0)
# |+> is an eigenstate of X with eigenvalue +1, so nothing is kicked back.
# Both branches keep a + sign, and the interference always sends us to 0.
```

The algorithm needs an eigenvector of X with eigenvalue $-1$. Kickback with eigenvalue $+1$ is still kickback, but it kicks back nothing.
</details>

Further reading: D. Deutsch, [Quantum theory, the Church-Turing principle and the universal quantum computer](https://doi.org/10.1098/rspa.1985.0070), Proc. R. Soc. A 400 (1985).

## Chapter 5. Going to $n$ bits: the promise problem

Deutsch's speed-up is 2 queries versus 1. Real, but not impressive. In 1992 Deutsch and Richard Jozsa asked what happens with more input bits.

Now $f:\{0,1\}^n\to\{0,1\}$ takes $n$ bits. There are $N = 2^n$ possible inputs. Most such functions are neither constant nor balanced, so the problem comes with a **promise**.

**The Deutsch-Jozsa problem.** We are *promised* that $f$ is either constant (same output on all $N$ inputs) or balanced (output 0 on exactly $N/2$ inputs and 1 on the other $N/2$). Decide which.

### How many classical queries?

Suppose a classical algorithm queries $f$ and keeps seeing 0. When can it stop?

- If it ever sees two different outputs, $f$ is balanced. Done.
- If it has seen 0 every time, it cannot rule out "balanced" until it has seen *more than half* of the inputs. An unlucky balanced function might hide all its 1s in the inputs not yet queried.

So a deterministic classical algorithm needs $N/2 + 1 = 2^{n-1} + 1$ queries in the worst case. **Exponential in $n$.**

The quantum algorithm will use **one** query, for every $n$, with certainty.

> **Physics note.** Deutsch and Jozsa published this in the *Proceedings of the Royal Society* in 1992, under the title *Rapid solution of problems by quantum computation*. It was the first example of an exponential separation between quantum and *deterministic* classical computation. The separation is real but narrow, and the next code cell shows why: a classical algorithm that is allowed a small chance of error is very cheap. Bernstein and Vazirani (1993) and Simon (1994) sharpened the separation step by step, and Simon's algorithm directly inspired Peter Shor's factoring algorithm the same year.

Here is the classical algorithm, run against a deliberately unlucky balanced function that hides its 1s at the end.

In [ ]:
def classical_dj(f, n):
    queries, first = 0, None
    for x in range(2**n):
        fx = f(x); queries += 1
        if first is None:
            first = fx
        elif fx != first:
            return "balanced", queries
        if queries == 2**(n - 1) + 1:                 # more than half agreed: must be constant
            return "constant", queries
    return "constant", queries

for n in [2, 4, 8, 12]:
    unlucky = lambda x, n=n: 0 if x < 2**(n - 1) else 1   # balanced, but 0s come first
    verdict, q = classical_dj(unlucky, n)
    print(f"n = {n:2d}:  {verdict} after {q:5d} queries   (N = {2**n})")

### What if a little error is allowed?

A **randomised** classical algorithm can query $k$ random inputs and answer "constant" if all $k$ agree. For a balanced function, each random query is a fair coin, so the chance that all $k$ agree (and we wrongly say "constant") is about $2\cdot(1/2)^k = 2^{1-k}$. With $k = 10$ queries the error is below 0.2%, **regardless of $n$**.

In [ ]:
ns = np.arange(1, 21)
ks = np.arange(1, 21)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.semilogy(ns, 2.0**(ns - 1) + 1, "o-", label="classical, deterministic: $2^{n-1}+1$")
ax1.semilogy(ns, np.ones_like(ns), "s-", label="quantum (Deutsch-Jozsa): 1")
ax1.set_xlabel("input bits n"); ax1.set_ylabel("queries (worst case)")
ax1.set_title("Exact answer"); ax1.legend(); ax1.grid(alpha=.3)

ax2.semilogy(ks, 2.0**(1 - ks), "o-", color="C2")
ax2.set_xlabel("random classical queries k"); ax2.set_ylabel("probability of a wrong answer")
ax2.set_title("Randomised classical algorithm, any n"); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

So be honest when you teach this: Deutsch-Jozsa beats *exact* classical algorithms exponentially, but a randomised classical algorithm with a tiny error rate is almost as good. Its value is as the cleanest possible demonstration of the mechanism (one query, interference, global property), which Bernstein-Vazirani, Simon and Shor then turn into stronger results.

### Exercise 5: counting queries

**Goal.** For $n = 10$ input bits, how many queries does the best deterministic classical algorithm need in the worst case? Store the number in `ex5_queries`.

<details>
<summary>Hint</summary>

$N/2 + 1$ with $N = 2^{10}$.
</details>

In [ ]:
ex5_queries = None

In [ ]:
check("worst-case deterministic queries for n = 10", lambda: ex5_queries == 513)

<details>
<summary>Solution</summary>

```python
ex5_queries = 2**9 + 1     # 513
```

After 512 agreeing answers we still cannot exclude a balanced function whose 512 ones are exactly the other half. The 513th query settles it.
</details>

Further reading: D. Deutsch and R. Jozsa, [Rapid solution of problems by quantum computation](https://doi.org/10.1098/rspa.1992.0167), Proc. R. Soc. A 439 (1992).

## Chapter 6. The mathematics of Deutsch-Jozsa

The circuit is the obvious generalisation of Deutsch's: $n$ input qubits instead of one.

$$|0\rangle^{\otimes n}|1\rangle \xrightarrow{H^{\otimes(n+1)}} \xrightarrow{U_f} \xrightarrow{H^{\otimes n}\otimes I} \text{measure the } n \text{ inputs.}$$

To analyse it we need one identity about Hadamards on many qubits.

### The Hadamard transform

For one qubit, $H|0\rangle = \frac{|0\rangle+|1\rangle}{\sqrt2}$ and $H|1\rangle = \frac{|0\rangle-|1\rangle}{\sqrt2}$. Both are captured by

$$H|b\rangle = \frac{1}{\sqrt2}\sum_{c\in\{0,1\}}(-1)^{bc}\,|c\rangle .$$

The minus sign appears only when $b = c = 1$. Take the tensor product over $n$ qubits, $x = x_{n-1}\dots x_0$ and $z = z_{n-1}\dots z_0$:

$$H^{\otimes n}|x\rangle = \frac{1}{\sqrt N}\sum_{z\in\{0,1\}^n}(-1)^{x\cdot z}\,|z\rangle, \qquad x\cdot z = x_0z_0 \oplus x_1z_1\oplus\dots\oplus x_{n-1}z_{n-1}.$$

The sign of each term is the **parity** of the bits that $x$ and $z$ have in common. Two special cases are worth remembering. For $x = 0$, every sign is $+$ and we get the uniform superposition. For $z = 0$, every sign is $+$ too: **every** $H^{\otimes n}|x\rangle$ has amplitude $+1/\sqrt N$ on $|0\dots0\rangle$.

Let us check the identity numerically before using it.

In [ ]:
def hadamard_transform_formula(x, n):
    N = 2**n
    return np.array([(-1)**bin(x & z).count("1") for z in range(N)]) / np.sqrt(N)

n = 3
ok = True
for x in range(2**n):
    qc = QuantumCircuit(n)
    qc.prepare_state(Statevector.from_int(x, 2**n))
    qc.h(range(n))
    ok &= np.allclose(Statevector(qc).data, hadamard_transform_formula(x, n))
print("H^n |x> matches the (-1)^(x.z) formula for every x on 3 qubits?", ok)

### Following the state

**Step 1.** After the first layer of Hadamards,

$$|\psi_1\rangle = \frac{1}{\sqrt N}\sum_{x}|x\rangle\otimes|-\rangle .$$

**Step 2.** One query. By phase kickback, every branch picks up its sign:

$$|\psi_2\rangle = \frac{1}{\sqrt N}\sum_{x}(-1)^{f(x)}|x\rangle\otimes|-\rangle .$$

All $N$ values of $f$ are now encoded in the state, one sign per branch.

**Step 3.** Hadamards on the inputs. Using the identity on each $|x\rangle$ and swapping the order of the sums,

$$|\psi_3\rangle = \sum_{z}\left[\frac{1}{N}\sum_{x}(-1)^{f(x)\,\oplus\,x\cdot z}\right]|z\rangle\otimes|-\rangle .$$

**Step 4.** Look only at the amplitude of $z = 0\dots0$, where $x\cdot z = 0$ for every $x$:

$$\boxed{\;a_{0\dots0} = \frac{1}{N}\sum_{x}(-1)^{f(x)}\;}$$

This is the **average of the signs**.

- **Constant:** all $N$ signs are equal, so $a_{0\dots0} = \pm1$. Since probabilities sum to one, every other amplitude is zero. We measure $0\dots0$ **with certainty**.
- **Balanced:** $N/2$ signs are $+1$ and $N/2$ are $-1$. They cancel exactly, so $a_{0\dots0} = 0$. We **never** measure $0\dots0$.

So the rule is: **all zeros means constant, anything else means balanced.** One query, no error.

For $n = 1$ this reduces exactly to Chapter 4: $a_0 = \frac{(-1)^{f(0)}+(-1)^{f(1)}}{2}$. Deutsch's algorithm is the special case, and Jozsa's contribution is to notice that the same cancellation works for averages over any number of inputs.

> **Predict.** For a balanced function, we know we will not see $0\dots0$. Will we see one particular other string, or many? Does it depend on *which* balanced function? Chapter 8 answers this with experiments.

### Exercise 6: signs of the Hadamard transform

**Goal.** Without using Qiskit, write down the eight signs $(-1)^{x\cdot z}$ for $x = 5 = 101$ and $z = 0, 1, \dots, 7$, as a list `ex6_signs` of $+1$ and $-1$.

<details>
<summary>Hint</summary>

$x = 101$ has bits 0 and 2 set. $x\cdot z$ is the parity of $z_0 + z_2$. So the sign is $-1$ exactly when one, but not both, of $z_0$ and $z_2$ is 1. For $z = 1 = 001$: $z_0 = 1, z_2 = 0$, so $-1$.
</details>

In [ ]:
ex6_signs = None

In [ ]:
check("signs of H^3 |101>", lambda: np.allclose(ex6_signs, hadamard_transform_formula(5, 3) * np.sqrt(8)))

<details>
<summary>Solution</summary>

```python
ex6_signs = [+1, -1, +1, -1, -1, +1, -1, +1]
```

| $z$ | 000 | 001 | 010 | 011 | 100 | 101 | 110 | 111 |
|---|---|---|---|---|---|---|---|---|
| $z_0\oplus z_2$ | 0 | 1 | 0 | 1 | 1 | 0 | 1 | 0 |
| sign | + | − | + | − | − | + | − | + |

The pattern only depends on bits 0 and 2 of $z$; bit 1 is irrelevant because $x_1 = 0$.
</details>

Further reading: chapter 1.4.4 of Nielsen and Chuang presents Deutsch-Jozsa in this notation, with the qubits in big-endian order.

## Chapter 7. Building $n$-bit oracles

To test the algorithm we need oracles for $n$-bit constant and balanced functions. We use three families.

**Constant.** `const_0` is the empty circuit; `const_1` is an X on the output qubit.

**Balanced, linear.** For a nonzero bitstring $s$, the function $f_s(x) = s\cdot x$ (the parity of the bits of $x$ selected by $s$) is balanced. Its oracle is one CNOT from each selected input qubit into the output, because each CNOT XORs one $x_i$ into $y$. Adding an X on the output gives $s\cdot x\oplus1$, also balanced.

**Balanced, arbitrary.** Any truth table can be turned into an oracle by flipping the output once for every input with $f(x) = 1$. To flip only on input $x$, temporarily turn every 0-bit of $x$ into a 1 with X gates, apply a multi-controlled X, and undo the X gates. This costs one multi-controlled gate per 1 in the table, so it is exponentially expensive, but it works for any function and is fine for small $n$.

> **Why $s\cdot x$ is balanced.** Pick any bit $i$ with $s_i = 1$. Flipping $x_i$ flips $s\cdot x$. So the inputs pair up, $x$ with $x\oplus e_i$, and each pair contains exactly one 0 and one 1.

In [ ]:
def constant_oracle(n, value):
    qc = QuantumCircuit(n + 1, name=f"const_{value}")
    if value:
        qc.x(n)
    return qc


def linear_oracle(n, s, flip=False):
    # Balanced oracle for f(x) = s . x  (XOR 1 if flip). s is a bitstring in Qiskit order: s[-1] is bit 0.
    assert len(s) == n and "1" in s
    qc = QuantumCircuit(n + 1, name=f"s.x (s={s})")
    for i in range(n):
        if s[n - 1 - i] == "1":
            qc.cx(i, n)
    if flip:
        qc.x(n)
    return qc


def oracle_from_table(table, n):
    # Oracle for an arbitrary truth table [f(0), ..., f(2^n - 1)].
    qc = QuantumCircuit(n + 1, name="U_f")
    for x, fx in enumerate(table):
        if fx:
            zeros = [i for i in range(n) if not (x >> i) & 1]
            if zeros: qc.x(zeros)
            qc.mcx(list(range(n)), n)
            if zeros: qc.x(zeros)
    return qc


def random_balanced_table(n, seed=None):
    rng = np.random.default_rng(seed)
    table = np.zeros(2**n, dtype=int)
    table[rng.choice(2**n, size=2**(n - 1), replace=False)] = 1
    return list(table)


n = 3
examples = {
    "const_1":            constant_oracle(n, 1),
    "s . x, s = 101":     linear_oracle(n, "101"),
    "random balanced":    oracle_from_table(random_balanced_table(n, seed=7), n),
}
for name, orc in examples.items():
    table = truth_table(orc, n)
    print(f"{name:18s} truth table {table}   number of 1s = {sum(table)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.2), gridspec_kw={"width_ratios": [1, 1.3, 4]})
for ax, (name, orc) in zip(axes, examples.items()):
    orc.draw("mpl", ax=ax, fold=-1)
    ax.set_title(name)
plt.show()

The linear oracle needs only two CNOTs. The random balanced oracle needs four multi-controlled gates plus their X sandwiches. Remember the difference when we get to noise in Chapter 9.

> **Predict.** `oracle_from_table` uses one multi-controlled X for every 1 in the table. For a random balanced function on 10 bits, how many is that?

### Exercise 7: a parity oracle

**Goal.** Build an oracle `ex7_oracle` on 3 input qubits (plus the output qubit 3) for $f(x) = x_0\oplus x_2$, using only CNOT gates. Then check that it is balanced.

<details>
<summary>Hint</summary>

This is $s\cdot x$ for which $s$? You can call `linear_oracle`, but try writing the two CNOTs yourself first.
</details>

In [ ]:
ex7_oracle = None

In [ ]:
check("ex7_oracle computes x0 XOR x2",
      lambda: truth_table(ex7_oracle, 3) == [(x & 1) ^ ((x >> 2) & 1) for x in range(8)])
check("it is balanced", lambda: sum(truth_table(ex7_oracle, 3)) == 4)

<details>
<summary>Solution</summary>

```python
ex7_oracle = QuantumCircuit(4)
ex7_oracle.cx(0, 3)
ex7_oracle.cx(2, 3)
# equivalently: ex7_oracle = linear_oracle(3, "101")
```

Notice that this is the *same* function as the middle example above. In Qiskit order, $s = $ `"101"` selects bits 0 and 2.
</details>

Further reading: [Multi-controlled gates in the circuit library](https://quantum.cloud.ibm.com/docs/en/api/qiskit/circuit_library), and Essentials Chapter 10 for building logic from reversible gates.

## Chapter 8. Running Deutsch-Jozsa, and what happens when the promise breaks

The circuit `dj_circuit` from Chapter 4 already handles any $n$. We only need a function that turns the measurement into a verdict.

In [ ]:
def dj_verdict(counts, n):
    zeros = "0" * n
    if counts.get(zeros, 0) == sum(counts.values()):
        return "constant"
    if zeros not in counts:
        return "balanced"
    return "neither (promise broken)"

n = 3
display(dj_circuit(examples["s . x, s = 101"], n).draw("mpl", fold=-1))

> **Predict.** For each of the three example oracles, which bitstrings will appear?

In [ ]:
all_counts = {}
for name, orc in examples.items():
    counts = sampler.run([dj_circuit(orc, n)], shots=1000).result()[0].data.c.get_counts()
    all_counts[name] = counts
    print(f"{name:18s} {dj_verdict(counts, n):10s}  {counts}")

plot_histogram(list(all_counts.values()), legend=list(all_counts.keys()),
               title="Deutsch-Jozsa on three oracles, one query each", figsize=(10, 4))

Three things to notice.

1. The constant oracle returns `000` every time.
2. The **linear** balanced oracle returns a **single** string, `101`, which is exactly $s$. That is not a coincidence, and it is the subject of Chapter 10.
3. The **random** balanced oracle returns several strings, but never `000`.

The general amplitude formula from Chapter 6, $a_z = \frac1N\sum_x(-1)^{f(x)\oplus x\cdot z}$, explains all three: it is a correlation between $f$ and the parity function $x\cdot z$. A linear $f$ correlates perfectly with exactly one parity; a random one correlates partly with several.

### Larger $n$, still one query

The query count does not grow. We run random balanced and constant functions for $n = 1$ to $6$ and confirm the verdict every time.

In [ ]:
rng = np.random.default_rng(2026)
print(" n | function        | verdict   | correct?")
print("---+-----------------+-----------+---------")
for n in range(1, 7):
    for kind in ["constant", "balanced"]:
        if kind == "constant":
            orc = constant_oracle(n, int(rng.integers(2)))
        else:
            orc = oracle_from_table(random_balanced_table(n, seed=int(rng.integers(10**6))), n)
        counts = sampler.run([dj_circuit(orc, n)], shots=200).result()[0].data.c.get_counts()
        v = dj_verdict(counts, n)
        print(f"{n:2d} | {kind:15s} | {v:9s} | {v == kind}")

### Breaking the promise

What does the circuit do with a function that is neither constant nor balanced, like `and_` from Exercise 1? The formula from Chapter 6 still holds. If $f$ has $k$ ones out of $N$ inputs, the sum of signs is $(N - k) - k$, so

$$P(0\dots0) = \left(1 - \frac{2k}{N}\right)^2 .$$

This is 1 for $k = 0$ or $k = N$ (constant), 0 for $k = N/2$ (balanced), and something in between otherwise. For a function outside the promise the algorithm gives a random answer, and with one shot you cannot tell that anything went wrong. That is why the promise matters.

In [ ]:
and_oracle = QuantumCircuit(3, name="U_and")
and_oracle.ccx(0, 1, 2)                                  # same as the Exercise 2 solution
counts = sampler.run([dj_circuit(and_oracle, 2)], shots=4000).result()[0].data.c.get_counts()
print("AND function, k = 1 one out of N = 4 inputs")
print("P(00) measured:", counts.get("00", 0) / 4000, "   theory:", (1 - 2 * 1 / 4)**2)
plot_histogram(counts, title="Deutsch-Jozsa on AND (promise broken)")

### Interactive: slide from constant to balanced

$N = 8$. The slider sets how many inputs return 1. The circuit is built from the truth table and simulated exactly.

In [ ]:
@interact(k=IntSlider(min=0, max=8, step=1, value=2, description="ones k"))
def promise_slider(k):
    n, N = 3, 8
    table = [1] * k + [0] * (N - k)
    sv = Statevector(dj_circuit(oracle_from_table(table, n), n).remove_final_measurements(inplace=False))
    p = sv.probabilities(list(range(n)))
    kind = "constant" if k in (0, N) else ("balanced" if k == N // 2 else "neither")
    print(f"k = {k}  ({kind})    P(000) = {p[0]:.3f}    theory (1 - 2k/N)^2 = {(1 - 2 * k / N)**2:.3f}")
    plt.figure(figsize=(7, 3))
    plt.bar([format(z, "03b") for z in range(N)], p)
    plt.ylim(0, 1); plt.ylabel("probability"); plt.title(f"Deutsch-Jozsa output, k = {k}")
    plt.show()

### Exercise 8: predicting a broken promise

**Goal.** A three-bit function has exactly two 1s in its truth table. Without simulating, compute the probability that Deutsch-Jozsa returns `000` and store it as `ex8_p`.

<details>
<summary>Hint</summary>

$N = 8$, $k = 2$. Use $(1 - 2k/N)^2$.
</details>

In [ ]:
ex8_p = None

In [ ]:
def _ex8():
    sv = Statevector(dj_circuit(oracle_from_table([0, 1, 0, 0, 0, 0, 1, 0], 3), 3).remove_final_measurements(inplace=False))
    return np.isclose(ex8_p, sv.probabilities([0, 1, 2])[0])

check("P(000) for a function with two 1s out of eight", _ex8)

<details>
<summary>Solution</summary>

```python
ex8_p = (1 - 2 * 2 / 8)**2      # 0.25
```

The six $+1$ signs and two $-1$ signs average to $4/8 = 1/2$, and the probability is its square. It does not matter *which* two inputs give 1.
</details>

Further reading: the Deutsch-Jozsa lesson in [Fundamentals of quantum algorithms](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms).

## Chapter 9. Noise and real devices

On ideal hardware the answer is certain. On a real device every gate adds error, and the two kinds of oracle from Chapter 7 have very different costs. We run the constant oracle (no gates at all) and the "all ones" linear oracle ($n$ CNOTs into one qubit) on the `FakeSherbrooke` snapshot from Essentials Chapter 8, for growing $n$.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

fake = FakeSherbrooke()
noisy_sim = AerSimulator.from_backend(fake)
pm = generate_preset_pass_manager(backend=fake, optimization_level=3, seed_transpiler=3)

ns = list(range(1, 8))
p_const, p_bal, p_exact, twoq = [], [], [], []
for n in ns:
    for kind, orc, store in [("constant", constant_oracle(n, 0), p_const),
                             ("balanced", linear_oracle(n, "1" * n), p_bal)]:
        isa = pm.run(dj_circuit(orc, n))
        counts = noisy_sim.run(isa, shots=4000, seed_simulator=1).result().get_counts()
        right = counts.get("0" * n, 0) if kind == "constant" else 4000 - counts.get("0" * n, 0)
        store.append(right / 4000)
        if kind == "balanced":
            p_exact.append(counts.get("1" * n, 0) / 4000)          # the exact string s = 11...1
            ops = isa.count_ops(); twoq.append(ops.get("ecr", 0) + ops.get("cz", 0))

plt.figure(figsize=(7, 3.5))
plt.plot(ns, p_const, "o-", label="constant oracle")
plt.plot(ns, p_bal, "s-", label="balanced, scored as 'not all zeros'")
plt.plot(ns, p_exact, "^-", label="balanced, scored as 'exactly s = 11...1'")
plt.axhline(1, ls="--", c="gray")
plt.xlabel("input bits n"); plt.ylabel("P(correct)"); plt.ylim(0.5, 1.02)
plt.title("Deutsch-Jozsa on a simulated IBM Sherbrooke"); plt.legend(); plt.grid(alpha=.3); plt.show()
print("two-qubit gates in the balanced circuit after transpiling:", dict(zip(ns, twoq)))

Three curves, three lessons.

- The **constant** case slowly degrades with $n$, almost entirely from **readout error**: every one of the $n$ qubits must be read as 0, and each has a small chance of being misread.
- The **balanced** case scored as "not all zeros" looks almost perfect. That is a property of the scoring, not of the device: noise scatters the answer over many wrong strings, and it rarely lands on exactly `0...0`. The verdict "balanced" is still correct, so for the yes/no question this is fair.
- Scored as "returned exactly $s$", the same balanced circuit falls off much faster. The transpiler has to route $n$ CNOTs onto one qubit of a sparse chip, so the two-qubit gate count (printed below the plot) grows faster than $n$. This is the honest measure of how well the *circuit* runs, and it is the one that matters for Bernstein-Vazirani in Chapter 10.

> **Predict.** Which verdict would a completely noisy device (every outcome equally likely) give? For $n = 7$, what is the chance it says "balanced"? What does that tell you about using "not all zeros" as the success test?

### Running on IBM Quantum hardware (optional)

The same pattern as Essentials Chapter 9. Off by default.

In [ ]:
RUN_ON_HARDWARE = False        # set to True once your account is saved (Essentials, Chapter 9)

if RUN_ON_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RuntimeSampler

    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)

    n = 3
    circuits = [dj_circuit(constant_oracle(n, 1), n), dj_circuit(linear_oracle(n, "101"), n)]
    isa_hw = generate_preset_pass_manager(backend=backend, optimization_level=3).run(circuits)

    sampler_hw = RuntimeSampler(mode=backend)
    sampler_hw.options.dynamical_decoupling.enable = True
    sampler_hw.options.twirling.enable_gates = True

    job = sampler_hw.run(isa_hw, shots=2000)
    print("Job ID:", job.job_id())
    res = job.result()
    hw = [r.data.c.get_counts() for r in res]
    display(plot_histogram(hw, legend=["constant", "balanced s=101"], title=f"Deutsch-Jozsa on {backend.name}"))

### Exercise 9: which oracle survives noise better?

**Goal.** On six input bits, compare two balanced linear oracles on the noisy simulator: $s = $ `"111111"` and $s = $ `"000001"`. Score each by the probability of measuring the **exact** string $s$ (not just "not all zeros"). Store the string with the higher score in `ex9_better`.

> **Predict** first: which one has fewer two-qubit gates after transpiling?

<details>
<summary>Hint</summary>

Reuse `pm`, `noisy_sim` and `linear_oracle`. Read `counts.get(s, 0) / shots` for each.
</details>

In [ ]:
ex9_better = None

In [ ]:
check("the oracle with fewer CNOTs survives noise better", lambda: ex9_better == "000001")

<details>
<summary>Solution</summary>

```python
scores = {}
for s in ["111111", "000001"]:
    isa = pm.run(dj_circuit(linear_oracle(6, s), 6))
    counts = noisy_sim.run(isa, shots=4000, seed_simulator=1).result().get_counts()
    scores[s] = counts.get(s, 0) / 4000
print(scores)
ex9_better = max(scores, key=scores.get)       # "000001"
```

`"000001"` uses one CNOT, `"111111"` six (more after routing). Same algorithm, same single query, but the cost of a query depends on how the oracle is built. In real applications the oracle is usually the most expensive part of the circuit.
</details>

Further reading: [Build noise models](https://quantum.cloud.ibm.com/docs/en/guides/build-noise-models) and [Error mitigation and suppression](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques).

## Chapter 10. A worked problem: Bernstein-Vazirani

In Chapter 8 the linear oracle $f(x) = s\cdot x$ returned exactly $s$. We now turn that observation into an algorithm, one decision at a time, the same way we built the half adder in Essentials Chapter 10. Before each code cell there is a question. Think first, then open the answer.

**The problem.** A function $f(x) = s\cdot x \bmod 2$ hides an unknown $n$-bit string $s$. Find $s$.

### Step 1. How would you do it classically?

Each query returns one bit. Which inputs would you query, and how many queries do you need?

<details>
<summary>Answer</summary>

Query the "one-hot" inputs $e_i = 0\dots010\dots0$. Then $f(e_i) = s_i$, one bit of $s$ per query, so $n$ queries in total. You cannot do better classically, because each query returns only one bit and $s$ has $n$ unknown bits.
</details>

In [ ]:
secret = "1011"
n = len(secret)
f = lambda x: bin(int(secret, 2) & x).count("1") % 2      # s . x mod 2, using integers

recovered = "".join(str(f(1 << i)) for i in reversed(range(n)))   # query e_{n-1}, ..., e_0
print(f"classical: {n} queries -> s = {recovered}")

### Step 2. What does the Deutsch-Jozsa circuit output for this $f$?

Use the general amplitude from Chapter 6,

$$a_z = \frac{1}{N}\sum_{x}(-1)^{s\cdot x\,\oplus\,x\cdot z}.$$

Simplify the exponent. When is the sum not zero?

<details>
<summary>Answer</summary>

$s\cdot x\oplus z\cdot x = (s\oplus z)\cdot x$. Now apply the Hadamard identity with $x$ and $z$ swapped: $\frac1N\sum_x(-1)^{w\cdot x}$ is 1 if $w = 0$ and 0 otherwise (for $w\neq0$ it is a balanced sum). So

$$a_z = \begin{cases}1 & z = s\\ 0 & \text{otherwise.}\end{cases}$$

**The circuit outputs $s$ with certainty.** In other words, $H^{\otimes n}$ maps the phase pattern $(-1)^{s\cdot x}$ back to $|s\rangle$, which is the Hadamard identity read backwards.
</details>

### Step 3. Do we need a new circuit?

<details>
<summary>Answer</summary>

No. It is exactly `dj_circuit` with the oracle `linear_oracle(n, s)`. Only the interpretation of the output changes: we read the whole bitstring, not just "all zeros or not".
</details>

In [ ]:
def bernstein_vazirani(oracle, n):
    counts = sampler.run([dj_circuit(oracle, n)], shots=100).result()[0].data.c.get_counts()
    return max(counts, key=counts.get), counts

bv = dj_circuit(linear_oracle(n, secret), n)
display(bv.draw("mpl", fold=-1))

found, counts = bernstein_vazirani(linear_oracle(n, secret), n)
print(f"quantum: 1 query -> s = {found}   (counts: {counts})")

### Step 4. Test every possible secret

A circuit is not solved until it is tested. For $n = 4$ there are 15 nonzero secrets (and $s = 0$, which gives a constant function).

In [ ]:
n = 4
all_ok = True
for s_int in range(1, 2**n):
    s = format(s_int, f"0{n}b")
    found, _ = bernstein_vazirani(linear_oracle(n, s), n)
    all_ok &= (found == s)
print(f"All {2**n - 1} secrets recovered with one query each:", all_ok)

### Step 5. Where did the bit ordering go?

We wrote the secret as a Qiskit-ordered string ($s_0$ on the right), and `linear_oracle` puts a CNOT on qubit $i$ when bit $i$ is set. The measured string is printed in the same order. So the output matches the secret character for character. If you ever see a reversed answer, check whether your oracle and your string use the same convention.

> **Physics note.** Ethan Bernstein and Umesh Vazirani introduced this problem in 1993, in the paper that defined the complexity class BQP. With one query instead of $n$, the gap is only polynomial. But they nested the problem recursively to get a superpolynomial separation even against randomised classical algorithms, closing the loophole we saw in Chapter 5. A year later Simon found an exponential separation, and Shor generalised Simon's idea into factoring.

**What we learned.**

- The Deutsch-Jozsa circuit does more than answer yes or no. For linear functions it hands back the full hidden string.
- Both algorithms use the same three moves: put the input in uniform superposition, write $f$ into signs with one query (phase kickback), and interfere with Hadamards so that the answer survives the measurement.
- Replacing the Hadamard transform by the **quantum Fourier transform** gives the same structure for periodic functions. That is the road to phase estimation and Shor's algorithm.

### Exercise 10: crack the mystery oracle

**Goal.** The oracle below hides a 6-bit secret, and also adds a constant 1 to make life harder: it computes $s\cdot x\oplus1$. Find $s$ with one query and store it as `ex10_s`. Then explain in a comment why the extra $\oplus1$ makes no difference.

In [ ]:
mystery = linear_oracle(6, format(53, "06b"), flip=True)     # no peeking at the number
mystery.name = "mystery"

ex10_s = None

In [ ]:
check("you found the hidden string", lambda: ex10_s == "110101")

<details>
<summary>Hint</summary>

`bernstein_vazirani(mystery, 6)`. For the explanation: $(-1)^{s\cdot x\oplus1} = -(-1)^{s\cdot x}$. What kind of phase is a $-1$ on *every* branch?
</details>

<details>
<summary>Solution</summary>

```python
ex10_s, counts = bernstein_vazirani(mystery, 6)     # '110101'
# The extra XOR 1 multiplies every branch by -1: a global phase, invisible to measurement.
```

The same argument shows why `const_0` and `const_1` give identical results in Deutsch-Jozsa.
</details>

**Stretch.** Bernstein-Vazirani is a natural candidate for real hardware because the oracle is just CNOTs. Run it on the noisy simulator for $s$ = `"1"*n` with $n = 2,\dots,8$ and plot the probability of the exact answer. Where does it fall below 50%?

Further reading: E. Bernstein and U. Vazirani, [Quantum complexity theory](https://doi.org/10.1137/S0097539796300921), SIAM J. Comput. 26 (1997); the Bernstein-Vazirani lesson in [Fundamentals of quantum algorithms](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms).

## Where we go next

| Topic | What it takes from this notebook |
|---|---|
| Simon's algorithm | Same "query in superposition, Hadamard, measure" pattern; finds a hidden XOR-period with exponentially fewer queries than any classical randomised algorithm |
| Quantum Fourier Transform | Replaces $H^{\otimes n}$; the Hadamard transform is the QFT over bits, the QFT works over integers mod $2^n$ |
| Quantum Phase Estimation | Phase kickback (Chapter 3) with controlled powers of a unitary instead of a single oracle |
| Grover search | A phase oracle $(-1)^{f(x)}$, applied many times, interleaved with an "inversion about the mean" built from Hadamards |
| Quantum error correction | Syndrome measurement is phase kickback onto an ancilla prepared in $|+\rangle$, exactly Chapter 3 |

## Quick reference

```python
# Bit oracle:   U_f |x>|y> = |x>|y XOR f(x)>       inputs on qubits 0..n-1, output on qubit n
# Phase oracle: with the output in |->, U_f acts as |x> -> (-1)^f(x) |x>

qc.x(n)                         # constant 1
qc.cx(i, n)                     # XOR x_i into the output (linear oracles)
qc.mcx(list(range(n)), n)       # flip the output only on |11...1>

dj_circuit(oracle, n)           # X, H on all, oracle, H on inputs, measure inputs
truth_table(oracle, n)          # read an oracle back into [f(0), ..., f(2^n - 1)]

# Reading the output:
#   all zeros            -> constant
#   anything else        -> balanced
#   for f(x) = s . x     -> the measured string IS s   (Bernstein-Vazirani)
#   broken promise       -> P(00...0) = (1 - 2k/N)^2
```

## Further reading

Original papers:
- D. Deutsch, [Quantum theory, the Church-Turing principle and the universal quantum computer](https://doi.org/10.1098/rspa.1985.0070) (1985)
- D. Deutsch and R. Jozsa, [Rapid solution of problems by quantum computation](https://doi.org/10.1098/rspa.1992.0167) (1992)
- R. Cleve, A. Ekert, C. Macchiavello and M. Mosca, [Quantum algorithms revisited](https://arxiv.org/abs/quant-ph/9708016) (1998)
- E. Bernstein and U. Vazirani, [Quantum complexity theory](https://doi.org/10.1137/S0097539796300921) (1993, journal version 1997)

IBM Quantum:
- [Fundamentals of quantum algorithms](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms), with lessons on the query model, Deutsch-Jozsa, Bernstein-Vazirani and Simon
- [Construct circuits](https://quantum.cloud.ibm.com/docs/en/guides/construct-circuits) and the [circuit library](https://quantum.cloud.ibm.com/docs/en/api/qiskit/circuit_library)

Textbook for the mathematics: Nielsen and Chuang, *Quantum Computation and Quantum Information*, section 1.4.